In [ ]:
#imports
import pandas as pd
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
import mpl_axes_aligner
from factor_analyzer.rotator import Rotator

In [ ]:
# Open the data
df = pd.read_csv('data/vwodata.csv')
print(df.head())

In [ ]:
# PCA: Scree plot
pca = PCA()  
pca_out = pca.fit_transform(df)
plt.plot(pca.explained_variance_, marker='o')
plt.xlabel('Principal Component')
plt.ylabel('Explained Variance Ratio')
plt.title('PCA Explained Variance')

In [ ]:

# Print(summary (explained variance ratio))
print(pd.DataFrame({
   'Explained Variance Ratio': pca.explained_variance_ratio_,
   'Cumulative Explained Variance': pca.explained_variance_ratio_.cumsum()
}))

In [ ]:
# Biplot function: Needs scores, loadings and pca (for variances)
def biplot(dfScores: pd.DataFrame, dfLoadings: pd.DataFrame, pca) -> None:
    
    # create figure and axis objects
    fig,ax = plt.subplots(figsize=(6,6))
    
    # plot the "scores" (pca coordinates for rows) 
    ax.scatter(dfScores.PC1.values,dfScores.PC2.values, s=5, color='b')  # s = size
   
    # we add explained variance in % to the axes
    explvar = 100 * pca.explained_variance_ratio_
    #set x-axis label
    ax.set_xlabel(f"PC1 ({explvar[0]:.1f}% explained var.)",fontsize=10)
    #set y-axis label
    ax.set_ylabel(f"PC2 ({explvar[1]:.1f}% explained var.)",fontsize=10)
        
    # create a second set of axes. That is: the plotting of the columns
    ax2 = ax.twinx().twiny()
    
    #setup font dictionary
    font = {'color':  'g',
            'weight': 'bold',
            'size': 12,
            }
    
    # make the loadings plot
    for col in dfLoadings.columns.values:
        #where do our loading vectors end?
        tipx = dfLoadings.loc['PC1',col]
        tipy = dfLoadings.loc['PC2',col]
        #draw the vector, and write label text for col
        ax2.arrow(0, 0, tipx, tipy, color = 'r', alpha = 1)
        ax2.text(tipx*1.05, tipy*1.05, col, fontdict = font, ha = 'center', va = 'center')
    
    #align x = 0 of ax and ax2 with the center of figure
    mpl_axes_aligner.align.xaxes(ax, 0, ax2, 0, 0.5)
    #align y = 0 of ax and ax2 with the center of figure
    mpl_axes_aligner.align.yaxes(ax, 0, ax2, 0, 0.5)
    
    # It is very important that the aspect ratio is 1:
    ax.set_aspect('equal', adjustable='datalim')
    ax2.set_aspect('equal', adjustable='datalim')

In [ ]:
# Collect PCA results in dataframes
dfScores   = pd.DataFrame(pca_out,columns=['PC'+str(i) for i in range(1,df.shape[1]+1)])
dfLoadings = pd.DataFrame(pca.components_,columns=df.columns,index=dfScores.columns)
#produce biplot
biplot(dfScores, dfLoadings, pca)

In [ ]:
# PCA with rotation
k = 3  # The number of components to be considered
pca = PCA(n_components=k, svd_solver="full", random_state=0)
pca.fit(df)

loadings = pca.components_.T

loadings_df = pd.DataFrame(loadings, index=df.columns, columns=["PC1", "PC2", "PC3"])
print(loadings_df)

rotator = Rotator(method="varimax")
loadings_rot = rotator.fit_transform(loadings)
R = rotator.rotation_

load_rot_df = pd.DataFrame(loadings_rot, index=df.columns, columns=["RPC1", "RPC2", "RPC3"])
print(load_rot_df)



In [ ]:
# Print "salient" rotated loadings
def pretty(df, cutoff):
     
    return df.where(df.abs() >= cutoff, other="")

In [ ]:
# We can select a cutoff for showing loadings
# Loadings that are larger (in absolute value) than this value (cut),  will be printed:
cut = 1/(len(load_rot_df)**0.5)  # Sum of squared elements is 1. So, if all same size, each is 1/sqrt(number of variables)
#cut = 0.1    # Some arbitrary value
# print(cut)

print("\nUnrotated Loadings:\n", pretty(loadings_df, cutoff=cut))

print("\nRotated Loadings:\n", pretty(load_rot_df, cutoff=cut))
print("\nRotation Matrix:\n", pd.DataFrame(R, index=["PC1", "PC2", "PC3"], columns=["RC1", "RC2", "RC3"]))
